In [1]:
sign = lambda x: -1 if x < 0 else 1

def tiles_in_line_prec2_yield(source_px: tuple, target_px: tuple, tile_res: int):
    """Get the tiles between 2 points in px. Tiles coordinates are returned in
    lazy mode and in the raster specified by tile_res.

    Tiles include both starting and ending tiles.
    """

    dx = target_px[0] - source_px[0]
    dy = target_px[1] - source_px[1]

    sx, sy = sign(dx), sign(dy)

    # Starting position
    x, y = source_px[0], source_px[1]


    if abs(dx) > abs(dy):
        x_incr = sx * tile_res
        y_incr = x_incr * dy/dx

        while sx*x < sx*target_px[0]:
            print(f'x:{x}, y:{y}, sx*x:{sx*x}, sx*target_px[0]:{sx*target_px[0]}')
            yield (x // tile_res, int(y // tile_res))
            x = x + x_incr
            y = y + y_incr

    else:
        y_incr = sy * tile_res
        x_incr = y_incr * dx/dy

        while sy*y < sy*target_px[1]:
            print(f'x:{x}, y:{y}, sy*y:{sy*y}, sy*target_px[1]:{sy*target_px[1]}')
            yield (int(x // tile_res), y // tile_res)
            x = x + x_incr
            y = y + y_incr


In [2]:
for tile in tiles_in_line_prec2_yield((200, 300), (10, 10), 64):
    print(tile)

x:200, y:300, sy*y:-300, sy*target_px[1]:-10
(3, 4)
x:158.0689655172414, y:236, sy*y:-236, sy*target_px[1]:-10
(2, 3)
x:116.13793103448278, y:172, sy*y:-172, sy*target_px[1]:-10
(1, 2)
x:74.20689655172416, y:108, sy*y:-108, sy*target_px[1]:-10
(1, 1)
x:32.27586206896554, y:44, sy*y:-44, sy*target_px[1]:-10
(0, 0)


In [3]:
def tiles_in_line_prec2(source_px: tuple, target_px: tuple, tile_res: int, precision: int) -> set:
    dx = target_px[0] - source_px[0]
    dy = target_px[1] - source_px[1]

    sign = lambda x: -1 if x < 0 else 1

    print(f'dx: {dx}, dy: {dy}')

    x, y = source_px[0], source_px[1]

    tiles = set()
    checkpoints = set()

    # Include the start
    tiles.add((x // tile_res, y // tile_res))
    checkpoints.add((x, y))

    if abs(dx) > abs(dy):
        print(f'Going horizontal')
        x_incr = sign(dx)*precision
        y_incr = x_incr * dy/dx

        while sign(dx)*x < sign(dx)*target_px[0]:
            x = x + x_incr
            y = y + y_incr
            tiles.add((x // tile_res, int(y // tile_res)))
            checkpoints.add((x, y))

    else:
        print(f'Going vertical')
        y_incr = sign(dy)*precision
        x_incr = y_incr * dx/dy

        while sign(dy)*y < sign(dy)*target_px[1]:
            x = x + x_incr
            y = y + y_incr
            tiles.add((int(x // tile_res), y // tile_res))
            checkpoints.add((x, y))

    # missing target in bottom, source on top

    # Remove the starting and finishing tile
    tiles.discard((source_px[0] // tile_res, source_px[1] // tile_res))
    tiles.discard((target_px[0] // tile_res, target_px[1] // tile_res))

    print(f'Tiles: {tiles}, Checkpoints: {checkpoints}')
    return tiles, checkpoints


## Define input parameters

In [4]:
TILE_RES = 64
PRECISION = 64
source_px_x, source_px_y = (200, 300)
target_px_x, target_px_y = (10, 10)


## Calculate the tiles

In [5]:
result, result_checkpoints = tiles_in_line_prec2( (source_px_x, source_px_y), (target_px_x, target_px_y), tile_res=TILE_RES, precision=PRECISION)
print(result)

dx: -190, dy: -290
Going vertical
Tiles: {(1, 2), (-1, -1), (1, 1), (2, 3)}, Checkpoints: {(158.0689655172414, 236), (74.20689655172416, 108), (200, 300), (32.27586206896554, 44), (116.13793103448278, 172), (-9.655172413793082, -20)}
{(1, 2), (-1, -1), (1, 1), (2, 3)}


## Display the result

In [6]:
from ipycanvas import MultiCanvas

multi_canvas = MultiCanvas(4, size=(640, 256))

# Basic grid
multi_canvas[0].fill_style = 'grey'
multi_canvas[0].stroke_style = 'black'
for y in range(5):
    for x in range(10):
        multi_canvas[0].stroke_rect(x*TILE_RES, y*TILE_RES, TILE_RES)

# Results without precision
multi_canvas[1].fill_style = 'grey'
multi_canvas[1].stroke_style = 'black'
for tile in result:
    multi_canvas[1].fill_rect(tile[0] * TILE_RES, tile[1] * TILE_RES, TILE_RES)

# Results with precision
multi_canvas[2].fill_style = 'blue'
multi_canvas[2].stroke_style = 'black'
for tile_prec in result_checkpoints:
    multi_canvas[2].fill_rect(tile_prec[0], tile_prec[1], 3)

# The original line
multi_canvas[3].fill_style = 'lime'
multi_canvas[3].stroke_style = 'green'

multi_canvas[3].begin_path()
multi_canvas[3].move_to(source_px_x, source_px_y)
multi_canvas[3].line_to(target_px_x, target_px_y)
multi_canvas[3].stroke()

multi_canvas


MultiCanvas()